# 01 - Data Understanding

## Objective

Notebook ini digunakan untuk memahami struktur awal dataset Customer 360
sebelum melakukan cleaning, standardization, atau entity resolution.

## Goals

1. Load dataset
2. Memahami struktur dataset
3. Mengetahui jumlah rows dan columns
4. Memeriksa nama kolom
5. Memeriksa data types
6. Memeriksa missing values
7. Memeriksa unique values
8. Memeriksa exact duplicates
9. Melihat sample records
10. Mengidentifikasi field yang berpotensi digunakan untuk entity resolution
11. Menentukan pertanyaan yang perlu dijawab sebelum matching

> Important:
> Tidak ada cleaning atau transformasi permanen pada notebook ini.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DATA_PATH = Path(r"data/raw/crm_50000_customers_dirty_v3.csv")

df = pd.read_csv(DATA_PATH)

In [ ]:
# Baca data
print(f"File: {DATA_PATH}")
print(f"Exists: {DATA_PATH.exists()}")

In [ ]:
# Ukuran
n_rows, n_cols = df.shape

print(f"Rows    : {n_rows:,}")
print(f"Columns : {n_cols}")

# Nama Kolom
print("Columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col}")

# Struktur
df.info

In [ ]:
# Data Type
dtype_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values
})

dtype_summary

# Sampel isi data
df.sample(10, random_state=42)


In [ ]:
# Cek Missing
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

missing_summary.round(2)

In [ ]:
# Deteksi Missing values
df.isna()

In [ ]:
# Cek Uniqe values
unique_summary = pd.DataFrame({
    "unique_count": df.nunique(dropna=True),
    "unique_percentage": (
        df.nunique(dropna=True) / len(df) * 100
    )
})

unique_summary = unique_summary.sort_values(
    "unique_count",
    ascending=False
)

unique_summary.round(2)

Persiapan ke Duplicate

In [ ]:
#exact
exact_duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicate_count:,}")

exact_duplicate_percentage = (
    exact_duplicate_count / len(df) * 100
)

print(f"Percentage: {exact_duplicate_percentage:.2f}%")

In [ ]:
df[df.duplicated(keep=False)].head(20)

Konsepnya:
Exact duplicate â‰  entity duplicate.

In [ ]:
if "customer_id" in df.columns:
    duplicated_customer_id = df["customer_id"].duplicated().sum()

    print(f"Duplicated customer_id: {duplicated_customer_id:,}")

if "customer_id" in df.columns:
    print(
        f"Unique customer_id: "
        f"{df['customer_id'].nunique(dropna=True):,}"
    )

In [ ]:
cardinality = pd.DataFrame({
    "column": df.columns,
    "rows": len(df),
    "unique": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ],
    "unique_ratio": [
        df[col].nunique(dropna=True) / len(df)
        for col in df.columns
    ]
})

cardinality = cardinality.sort_values(
    "unique_ratio",
    ascending=False
)

cardinality.round(4)

In [ ]:
for col in df.columns:
    unique_count = df[col].nunique(dropna=True)

    if unique_count <= 20:
        print(f"\n{'=' * 50}")
        print(f"{col}")
        print(f"{'=' * 50}")
        print(df[col].value_counts(dropna=False))

## Initial Entity Resolution Field Assessment

Pada tahap ini kita belum menentukan algoritma matching.

Field akan dikategorikan berdasarkan karakteristik awal:

- Potential identifier
- Potential strong matching field
- Potential supporting field
- Low-discriminative field
- Field yang perlu investigasi lebih lanjut

Penentuan final dilakukan setelah data quality dan duplicate pattern analysis.

In [ ]:
candidate_fields = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in [
        "id",
        "email",
        "phone",
        "name",
        "address",
        "dob",
        "date"
    ])
]

candidate_fields

In [ ]:
candidate_profile = cardinality[
    cardinality["column"].isin(candidate_fields)
].copy()

candidate_profile.round(4)

# Questions Before Matching

Sebelum masuk ke exact matching atau fuzzy matching, hasil profiling awal menjawab:

1. **Apakah `customer_id` benar-benar unik?** Tidak. Dataset memiliki 48.200 nilai unik dari 50.000 baris, sehingga terdapat 1.800 baris dengan `customer_id` yang berulang. Tidak ada nilai missing.
2. **Apakah terdapat exact duplicate records?** Ya, terdapat 1.021 baris exact duplicate atau sekitar 2,04% dari seluruh baris.
3. **Seberapa banyak missing value pada setiap field?** Missing value hanya ditemukan pada `email`, yaitu 1.040 baris atau 2,08%. Field lainnya tidak memiliki missing value berdasarkan `isna()`.
4. **Apakah email cukup lengkap dan cukup unik?** Cukup lengkap karena 48.960 dari 50.000 baris terisi (97,92%). Email memiliki 46.363 nilai unik. Seluruh email yang terisi lolos pemeriksaan format sederhana, tetapi keunikan belum berarti email bebas dari variasi huruf besar-kecil atau kesalahan semantik.
5. **Apakah phone number cukup lengkap dan cukup unik?** Lengkap karena tidak ada missing value. Terdapat 46.777 nilai unik (93,55%), tetapi panjang angka setelah karakter non-digit dihapus bervariasi antara 10 sampai 18 digit. Ini menunjukkan format nomor telepon belum konsisten dan perlu standardisasi sebelum matching.
6. **Seberapa banyak variasi pada nama?** Gabungan `first_name` dan `last_name` memiliki 42.585 kombinasi mentah dan 40.254 kombinasi setelah normalisasi sederhana. Selisih 2.331 menunjukkan adanya variasi penulisan atau tanda baca/spasi yang perlu diperiksa.
7. **Apakah terdapat indikasi typo atau karakter yang tidak konsisten?** Ada indikasi variasi pada nama dan telepon berdasarkan perubahan cardinality dan panjang nomor. Typo belum dapat dipastikan tanpa aturan validasi atau pemeriksaan pasangan record.
8. **Apakah format email konsisten?** Seluruh email terisi lolos regex format dasar `nama@domain.tld`. Konsistensi yang lebih dalam, seperti domain, case, spasi tersembunyi, atau typo domain, masih perlu diperiksa.
9. **Apakah format nomor telepon konsisten?** Tidak. Panjang digit bervariasi dari 10 hingga 18 digit dan jumlah format mentah berbeda dari jumlah format setelah normalisasi belum dianalisis per pola negara.
10. **Apakah address memiliki variasi format yang signifikan?** `address` memiliki 48.200 nilai unik, sama dengan cardinality `customer_id` dan `device_id(s)`. Normalisasi spasi sederhana tidak mengurangi jumlah nilai unik, sehingga variasi alamat tidak terbukti hanya disebabkan oleh spasi. Variasi tanda baca, singkatan, dan urutan komponen masih perlu dianalisis.
11. **Apakah dob memiliki format yang konsisten?** Semua 50.000 nilai berhasil diparse oleh `pandas.to_datetime`, tetapi format teks asli belum diuji secara eksplisit. `dob` memiliki 17.733 nilai unik, jadi konsistensi format dan kemungkinan tanggal yang sama dengan representasi berbeda masih perlu diperiksa.
12. **Apakah terdapat field yang terlalu banyak missing sehingga tidak reliable?** Tidak ada field dengan missing sangat tinggi. Hanya `email` yang missing 2,08%, sehingga email tetap usable tetapi perlu perlakuan khusus untuk baris kosong.
13. **Apakah terdapat field yang sangat discriminative?** `customer_id`, `address`, dan `device_id(s)` masing-masing memiliki 48.200 nilai unik (96,40%). `phone_number` memiliki 46.777 (93,55%) dan `email` 46.363 (92,73%). Field-field ini paling discriminative secara cardinality, tetapi belum otomatis dapat dianggap sebagai identifier yang benar.
14. **Apakah duplicate kemungkinan terjadi karena typo, formatting, missing value, atau kombinasi beberapa field?** Profiling membuktikan adanya exact duplicate, `customer_id` berulang, missing email, serta variasi nama dan telepon. Penyebab setiap duplicate belum dapat dipastikan; kemungkinan yang paling masuk akal adalah kombinasi exact duplication, variasi formatting, dan perubahan/ketidaklengkapan field.
15. **Apakah dataset menyediakan ground truth untuk mengevaluasi entity resolution?** Belum terlihat. Kolom yang tersedia tidak menunjukkan label pasangan duplicate atau `master_customer_id`. Hal ini perlu dikonfirmasi dari dokumentasi sumber data sebelum evaluasi supervised dilakukan.

Kesimpulan ini masih berada pada tahap profiling. Belum dilakukan cleaning, standardization, blocking, fuzzy matching, atau entity resolution.

# Initial Findings

## Dataset Structure

- Number of rows: **50,000**
- Number of columns: **14**
- Data types: seluruh kolom terbaca sebagai `str`; kolom tanggal (`dob` dan `signup_date`) belum dikonversi ke tipe datetime.

## Data Quality

- Missing values: hanya `email` yang memiliki missing value, yaitu **1.040 baris (2,08%)**. Kolom lain tidak memiliki missing value.
- Exact duplicate rows: **1.021 (2,04%)**
- High-cardinality fields: `customer_id`, `address`, dan `device_id(s)` masing-masing **48.200 unik (96,40%)**; `phone_number` **46.777 unik (93,55%)**; `email` **46.363 unik (92,73%)**.
- Low-cardinality fields: `gender` dan `source` masing-masing memiliki **3 nilai unik**; `state` memiliki **50**; `country` memiliki **243**. Field ini kurang cocok sebagai kunci tunggal untuk entity resolution.

## Potential Entity Resolution Fields

Berdasarkan profiling awal, field yang perlu diperiksa lebih lanjut:

- `email`: cukup lengkap dan cukup unik, tetapi 1.040 nilai missing.
- `phone_number`: cukup unik, tetapi memiliki panjang digit 10 sampai 18 sehingga perlu standardisasi.
- Kombinasi `first_name`, `last_name`, `dob`, dan `address`: dapat menjadi supporting fields karena variasi nama dan alamat perlu dibandingkan bersama.

## Important Observations

- `customer_id` tidak unik: 48.200 nilai unik untuk 50.000 baris, dengan 1.800 baris berulang.
- Terdapat 1.021 exact duplicate rows. Exact duplicate perlu dipisahkan dari entity duplicate karena record yang sama persis belum menjelaskan seluruh pola identitas pelanggan.
- `email` adalah satu-satunya field dengan missing value, sebesar 2,08%.
- `phone_number` memiliki format yang belum seragam berdasarkan panjang digit 10 sampai 18.
- Normalisasi sederhana pada gabungan nama mengurangi jumlah nilai unik dari 42.585 menjadi 40.254; ini mengindikasikan variasi penulisan yang perlu ditelusuri.
- Seluruh nilai `dob` dan `signup_date` berhasil diparse, tetapi format string asli belum dinilai secara eksplisit.

## Open Questions

1. Apakah pengulangan `customer_id` menunjukkan satu customer dengan banyak record, atau kesalahan pembuatan identifier?
2. Apakah variasi `phone_number`, nama, dan email berasal dari sumber data yang berbeda atau dari perubahan data pelanggan?
3. Apakah tersedia ground truth seperti `master_customer_id` atau label pasangan duplicate untuk mengukur precision dan recall matching?

## Next Step

Tahap berikutnya adalah Data Quality Assessment dan Duplicate Pattern Analysis.

Belum dilakukan:
- cleaning permanen
- standardization
- fuzzy matching
- blocking
- entity resolution